如果 graph 使用中断来实现人机交互，在时间旅行过程中中断总是会被重新触发。

包含中断的节点会重新执行，而 interrupt() 会暂停等待新的 Command(resume=...) 指令。

In [4]:
from langgraph.types import interrupt, Command
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
import operator


class State(TypedDict):
    value: Annotated[list[str], operator.add]

def ask_human(state: State):
    answer = interrupt("What is your name?")
    return {"value": [f"Hello, {answer}!"]}

def final_step(state: State):
    return {"value": ["Done"]}

graph = (
    StateGraph(State)
    .add_node("ask_human", ask_human)
    .add_node("final_step", final_step)
    .add_edge(START, "ask_human")
    .add_edge("ask_human", "final_step")
    .add_edge("final_step", END)
    .compile(checkpointer=InMemorySaver())
)

In [5]:
config = {"configurable": {"thread_id": "1"}}

# First run: hits interrupt
graph.invoke({"value": []}, config) # type: ignore

{'value': [],
 '__interrupt__': [Interrupt(value='What is your name?', id='f51783a69c0878b1fd78d209fbe33aa8')]}

In [6]:
# Resume with answer
graph.invoke(Command(resume="Alice"), config) # type: ignore

{'value': ['Hello, Alice!', 'Done']}

In [8]:
# Replay from before ask_human
history = list(graph.get_state_history(config)) # type: ignore
before_ask = [s for s in history if s.next == ("ask_human",)][-1]
before_ask.values, before_ask.next

({'value': []}, ('ask_human',))

In [9]:
replay_result = graph.invoke(None, before_ask.config)
# Pauses at interrupt — waiting for new Command(resume=...)
replay_result

{'value': [],
 '__interrupt__': [Interrupt(value='What is your name?', id='f51783a69c0878b1fd78d209fbe33aa8')]}

In [10]:
# Fork from before ask_human
fork_config = graph.update_state(before_ask.config, {"value": ["forked"]})
fork_result = graph.invoke(None, fork_config)
# Pauses at interrupt — waiting for new Command(resume=...)
fork_result

{'value': ['forked'],
 '__interrupt__': [Interrupt(value='What is your name?', id='a3306fb9d291d6b316d7931e72494136')]}

In [11]:
# Resume the forked interrupt with a different answer
graph.invoke(Command(resume="Bob"), fork_config)

{'value': ['forked', 'Hello, Bob!', 'Done']}